# Lab 3 Parte 1 — Solución CNN grietas (solo docente)

Referencia con hiperparámetros recomendados y respuestas teóricas sugeridas.
Los alumnos trabajan solo con `cnn_grietas_estructuras_alumno_ia.ipynb` + bitácora de prompts.


## Mapa del flujo típico de deep learning

Cada **Pregunta** del lab sigue el pipeline estándar de un proyecto DL. Úsalo como brújula:

| Paso DL | Pregunta | Qué haces aquí |
|---------|----------|----------------|
| 1. Problema y contexto | **1** | ¿Por qué CNN para grietas? |
| 2. Explorar datos | **2** | EDA, balance de clases, muestras |
| 3. Preprocesar / aumentar | **3** | Resize, normalize, augmentation |
| 4. Pipeline de datos | **4** | DataLoaders train/val |
| 5. Diseñar modelo | **5** | Arquitectura CNN (tu modelo pequeño) |
| 6. Modelo de referencia | **6** | Cargar checkpoint docente (sin reentrenar) |
| 7. Entrenar | **7** | Loss, optimizer, bucle corto en CPU |
| 8. Monitorear | **8** | Curvas train vs val (¿overfitting?) |
| 9. Comparar | **9** | Tu modelo vs docente (épocas e hiperparámetros) |
| 10. Evaluar | **10** | Métricas y matriz de confusión (docente) |
| 11. Analizar errores | **11** | Casos locales: aciertos y fallos |
| 12. Decisión ingeniería | **12** | ¿Desplegarías esto en obra? |

> **Idea clave:** el proceso es siempre el mismo; cambian los datos y la arquitectura (CNN aquí, LSTM en Parte 2).


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('.').resolve()))
from _verificar import (
    verificar_panorama_cnn, verificar_eda_dataset, verificar_augmentation,
    verificar_dataloaders, verificar_arquitectura, verificar_entrenamiento,
    verificar_curvas, verificar_metricas, verificar_casos_locales, resumen_seccion,
    verificar_modelo_docente_cargado, verificar_comparacion_docente,
)
from _modelo_cnn import CrackCNN, load_crack_cnn

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.transforms import (
    Resize, ToTensor, Normalize, RandomHorizontalFlip, RandomRotation, ColorJitter,
)
from PIL import Image

%matplotlib inline
sns.set_theme(style='whitegrid')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cpu')  # Codespaces / Binder: solo CPU
print(f"✅ Entorno listo | device={device} (entrenamiento corto alumno)")


## Contexto del dataset (Mendeley — grietas en hormigón)

| Clase | En obra significa… | Imágenes (subset) |
|-------|-------------------|-------------------|
| **Negative** | Superficie sin grieta visible | 800 train + 200 val |
| **Positive** | Superficie con grieta | 800 train + 200 val |

Imágenes **227×227** RGB; en el lab se redimensionan con `IMAGE_SIZE` para entrenar en CPU.

Detalle: [`data/DATOS.md`](data/DATOS.md).


## Pregunta 1 — Panorama CNN en inspección estructural

Las **CNN** aprenden filtros espaciales (bordes, texturas, grietas) a partir de píxeles.

### 📘 Subpreguntas
- **1.a** ¿Qué ventaja tiene una CNN frente a features manuales para grietas?
- **1.b** ¿Qué papel cumplen convolución y pooling?
- **1.c** ¿Por qué necesitamos miles de imágenes etiquetadas?


#### ✅ Respuesta sugerida

**1.a** La CNN aprende patrones locales (orientación de grieta, contraste) sin diseñar descriptores a mano.
**1.b** Convolución detecta patrones; pooling reduce dimensión y aporta invariancia a pequeños desplazamientos.
**1.c** Sin ejemplos variados (luz, textura) el modelo no generaliza a otras obras.


In [ ]:
COMPONENTES_CNN = ["convolución", "pooling", "ReLU", "flatten", "fully connected"]
print("Componentes CNN del laboratorio:")
for c in COMPONENTES_CNN:
    print(f"  · {c}")


In [ ]:
# --- Autoevaluación 1 ---
# Requiere (celda «Aquí coloca tu solución»): `COMPONENTES_CNN`
r = []
try:
    r = verificar_panorama_cnn(COMPONENTES_CNN)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('1 — Panorama CNN', r)


## Pregunta 2 — EDA del dataset

Antes de entrenar, revisa **balance**, **tamaños** y **ejemplos visuales**.

### 📘 Subpreguntas
- **2.a** ¿Están balanceadas las clases en train y val?
- **2.b** ¿Las imágenes tienen tamaño uniforme (227×227)?
- **2.c** ¿Qué diferencias visuales buscarías entre Positive y Negative?


#### ✅ Reflexión 2

Clases balanceadas; histograma confirma 227×227; mosaico muestra textura e iluminación.


In [ ]:
# --- PRE-ESCRITO: rutas y conteos ---
RUTA_DATOS = Path("data/cracks_subset")
if not RUTA_DATOS.is_dir():
    raise FileNotFoundError("Ejecuta primero: python _preparar_datos.py o bash labs/setup.sh")

conteos = {}
for split in ("train", "val"):
    conteos[split] = {}
    for cls in ("Negative", "Positive"):
        d = RUTA_DATOS / split / cls
        conteos[split][cls] = len(list(d.glob("*.jpg")))

class_names = ["Negative", "Positive"]
display(pd.DataFrame(conteos))


In [ ]:
N_EJEMPLOS_MOSAICO = 4
N_MUESTRAS_EDA = 100

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
x = np.arange(len(class_names))
width = 0.35
axes[0].bar(x - width / 2, [conteos['train'][c] for c in class_names], width, label='train')
axes[0].bar(x + width / 2, [conteos['val'][c] for c in class_names], width, label='val')
axes[0].set_xticks(x, class_names)
axes[0].set_title('Balance por clase'); axes[0].legend()

paths_eda = []
for cls in class_names:
    paths_eda.extend(sorted((RUTA_DATOS / 'train' / cls).glob('*.jpg')))
paths_eda = paths_eda[:N_MUESTRAS_EDA]
widths, heights = [], []
for p in paths_eda:
    with Image.open(p) as im:
        widths.append(im.width)
        heights.append(im.height)
axes[1].hist(widths, bins=15, alpha=0.7, label='ancho')
axes[1].hist(heights, bins=15, alpha=0.7, label='alto')
axes[1].set_title('Tamaños de imagen (muestra train)'); axes[1].legend()
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, N_EJEMPLOS_MOSAICO, figsize=(2 * N_EJEMPLOS_MOSAICO, 4))
for row, cls in enumerate(["Negative", "Positive"]):
    paths = sorted((RUTA_DATOS / 'train' / cls).glob('*.jpg'))[:N_EJEMPLOS_MOSAICO]
    for col, p in enumerate(paths):
        axes[row, col].imshow(Image.open(p))
        axes[row, col].set_title(cls if col == 0 else '')
        axes[row, col].axis('off')
plt.suptitle('Muestras de entrenamiento')
plt.tight_layout()
plt.show()


In [ ]:
# --- Autoevaluación 2 ---
# Requiere (celda «Aquí coloca tu solución»): `N_EJEMPLOS_MOSAICO`, `N_MUESTRAS_EDA`, `conteos`
r = []
try:
    r = verificar_eda_dataset(conteos, N_EJEMPLOS_MOSAICO, N_MUESTRAS_EDA)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('2 — EDA', r)


## Pregunta 3 — Transformaciones y data augmentation

En **train** añadimos variación artificial (flip, rotación, color); en **val** solo resize y normalize.

### 📘 Subpreguntas
- **3.a** ¿Por qué augmentamos solo en train?
- **3.b** ¿Qué riesgo tiene rotar demasiado una grieta?
- **3.c** ¿Por qué normalizamos con media 0.5?


#### ✅ Reflexión 3

Augmentation solo en train; val determinista para métricas comparables.


In [ ]:
IMAGE_SIZE = 128
AUG_ROTATION = 15
N_AUG_MOSTRADOS = 4

train_transform = transforms.Compose([
    RandomHorizontalFlip(),
    RandomRotation(AUG_ROTATION),
    ColorJitter(brightness=0.2, contrast=0.2),
    Resize((IMAGE_SIZE, IMAGE_SIZE)),
    ToTensor(),
    Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])
val_transform = transforms.Compose([
    Resize((IMAGE_SIZE, IMAGE_SIZE)),
    ToTensor(),
    Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

sample_path = sorted((RUTA_DATOS / 'train' / 'Positive').glob('*.jpg'))[0]
img_pil = Image.open(sample_path).convert('RGB')
fig, axes = plt.subplots(1, N_AUG_MOSTRADOS + 1, figsize=(2.2 * (N_AUG_MOSTRADOS + 1), 2.5))
axes[0].imshow(img_pil)
axes[0].set_title('original')
axes[0].axis('off')
for i in range(N_AUG_MOSTRADOS):
    aug = train_transform(img_pil)
    vis = aug.numpy().transpose(1, 2, 0) * 0.5 + 0.5
    axes[i + 1].imshow(vis.clip(0, 1))
    axes[i + 1].set_title(f'aug {i + 1}')
    axes[i + 1].axis('off')
plt.suptitle('Data augmentation en train')
plt.tight_layout()
plt.show()


In [ ]:
# --- Autoevaluación 3 ---
# Requiere (celda «Aquí coloca tu solución»): `IMAGE_SIZE`, `AUG_ROTATION`, `train_transform`, `val_transform`, `N_AUG_MOSTRADOS`
r = []
try:
    r = verificar_augmentation(IMAGE_SIZE, AUG_ROTATION, train_transform, val_transform, N_AUG_MOSTRADOS)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('3 — Augmentation', r)


## Pregunta 4 — DataLoaders

### 📘 Subpreguntas
- **4.a** ¿Por qué `shuffle=True` solo en train?
- **4.b** ¿Qué forma tiene un batch de imágenes (N×C×H×W)?


#### ✅ Reflexión 4

Shuffle en train evita orden de archivos; batch N×3×H×W.


In [ ]:
BATCH_SIZE = 32
train_ds = datasets.ImageFolder(RUTA_DATOS / 'train', transform=train_transform)
val_ds = datasets.ImageFolder(RUTA_DATOS / 'val', transform=val_transform)
class_names = train_ds.classes
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
print(f"Clases: {class_names} | train={len(train_ds)} | val={len(val_ds)}")


In [ ]:
# --- Autoevaluación 4 ---
# Requiere (celda «Aquí coloca tu solución»): `BATCH_SIZE`, `train_loader`, `val_loader`
r = []
try:
    r = verificar_dataloaders(BATCH_SIZE, train_loader, val_loader, image_size=IMAGE_SIZE)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('4 — DataLoaders', r)


## Pregunta 5 — Arquitectura CNN

### 📘 Subpreguntas
- **5.a** ¿Qué hace cada bloque Conv+ReLU+Pool?
- **5.b** ¿Por qué dropout antes de la salida?


#### ✅ Reflexión 5

Dos bloques conv capturan bordes y texturas; dropout reduce sobreajuste.


In [ ]:
N_FILTERS = 32
DROPOUT = 0.3

modelo = CrackCNN(n_filters=N_FILTERS, dropout=DROPOUT, use_batchnorm=False).to(device)
print(modelo)


In [ ]:
# --- Autoevaluación 5 ---
# Requiere (celda «Aquí coloca tu solución»): `modelo`, `N_FILTERS`, `DROPOUT`
r = []
try:
    r = verificar_arquitectura(modelo, N_FILTERS, DROPOUT)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('5 — Arquitectura', r)


#### ✅ Reflexión 6

CrossEntropy + Adam estándar; gap train-val indica overfitting.


## Pregunta 6 — Modelo docente (referencia)

El archivo `data/crack_cnn_best.pt` es el **mejor modelo** entrenado por el docente (más épocas, más filtros, BatchNorm). Lo cargamos **sin reentrenar** para comparar con tu modelo pequeño.

### 📘 Subpreguntas
- **6.a** ¿Qué hiperparámetros usa el docente (`model_meta.json`)?
- **6.b** ¿Por qué no reentrenamos este checkpoint en clase?


In [ ]:
# --- PRE-ESCRITO: cargar modelo docente (no reentrenar) ---
modelo_docente, meta_docente = load_crack_cnn(device=device)
hp_docente = meta_docente.get('hyperparameters', {})
N_EPOCHS_DOCENTE = hp_docente.get('epochs', 15)
_, acc_docente = eval_epoch(modelo_docente, val_loader, nn.CrossEntropyLoss(), device)
print(f"Modelo docente | acc val = {acc_docente:.3f}")
print(f"Hiperparámetros docente: {hp_docente}")


In [ ]:
# --- PRE-ESCRITO: funciones de entrenamiento y evaluación ---
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total

def eval_epoch(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return running_loss / total, correct / total

print("✅ Funciones train_one_epoch / eval_epoch listas.")


In [ ]:
N_EPOCHS = 3
LEARNING_RATE = 1e-3
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(modelo.parameters(), lr=LEARNING_RATE)
history = {k: [] for k in ['train_loss', 'val_loss', 'train_acc', 'val_acc']}
for epoch in range(N_EPOCHS):
    tl, ta = train_one_epoch(modelo, train_loader, criterion, optimizer, device)
    vl, va = eval_epoch(modelo, val_loader, criterion, device)
    history['train_loss'].append(tl)
    history['val_loss'].append(vl)
    history['train_acc'].append(ta)
    history['val_acc'].append(va)
    print(f"Época {epoch+1}/{N_EPOCHS} | loss {tl:.3f}/{vl:.3f} | acc {ta:.3f}/{va:.3f}")
acc_alumno = history['val_acc'][-1]
print(f"✅ Entrenamiento corto completado | acc_val={acc_alumno:.3f}")


## Pregunta 7 — Entrenamiento

### 📘 Subpreguntas
- **7.a** ¿Qué optimizador y loss usas para clasificación binaria?
- **7.b** ¿Cómo detectas overfitting con train vs val?


In [ ]:
# --- Autoevaluación 8 ---
# Requiere (celda «Aquí coloca tu solución»): `history`, `N_EPOCHS`, `LEARNING_RATE`
r = []
try:
    r = verificar_entrenamiento(history, N_EPOCHS, LEARNING_RATE)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('7 — Entrenamiento', r)


## Pregunta 8 — Curvas de entrenamiento

### 📘 Subpreguntas
- **7.a** ¿Val loss sube mientras train baja?
- **7.b** ¿Cuántas épocas serían suficientes en producción?


#### ✅ Reflexión 7

Si val loss sube, early stopping o más regularización.


In [ ]:
epochs = range(1, N_EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(epochs, history['train_loss'], label='train')
ax1.plot(epochs, history['val_loss'], label='val')
ax1.set_title('Pérdida'); ax1.legend()
ax2.plot(epochs, history['train_acc'], label='train')
ax2.plot(epochs, history['val_acc'], label='val')
ax2.set_title('Accuracy'); ax2.legend()
plt.tight_layout()
plt.show()


In [ ]:
# --- Autoevaluación 8 ---
# Requiere (celda «Aquí coloca tu solución»): `history`, `N_EPOCHS`
r = []
try:
    r = verificar_curvas(history, N_EPOCHS)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('8 — Curvas', r)


## Pregunta 9 — Comparación: tu modelo vs docente

Entrenaste **pocas épocas en CPU** con una CNN más pequeña. El docente usó **más tiempo y mejores hiperparámetros**.

### 📘 Subpreguntas
- **9.a** ¿Cuánto mejora el modelo docente respecto al tuyo?
- **9.b** ¿Qué cambió (épocas, `n_filters`, BatchNorm)?
- **9.c** ¿Vale la pena en obra pagar más cómputo por ese salto?


In [ ]:
acc_alumno = history['val_acc'][-1]
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(
    [f'Tu modelo\n({N_EPOCHS} ép., CPU)', f'Docente\n({N_EPOCHS_DOCENTE} ép., ref)'],
    [acc_alumno, acc_docente],
    color=['#3498db', '#27ae60'],
)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Accuracy validación')
ax.set_title('Comparación alumno vs modelo docente')
for i, v in enumerate([acc_alumno, acc_docente]):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center')
plt.tight_layout()
plt.show()
print(f"Tu modelo: acc={acc_alumno:.3f} | n_filters={N_FILTERS} | épocas={N_EPOCHS}")
print(f"Docente:   acc={acc_docente:.3f} | hp={hp_docente}")


In [ ]:
# --- Autoevaluación 9 (comparación) ---
r = []
try:
    r = verificar_comparacion_docente(acc_alumno, acc_docente, N_EPOCHS, N_EPOCHS_DOCENTE)
except NameError as err:
    print(f"❌ Ejecuta primero las celdas anteriores. Falta: {err}")
    r = [False]
resumen_seccion('9 — Comparación docente', r)


## Pregunta 10 — Métricas en validación (modelo docente)

### 📘 Subpreguntas
- **10.a** ¿Qué clase se confunde más?
- **10.b** ¿Un falso positivo (grieta) es más grave que un falso negativo?


#### ✅ Reflexión 8

Revisar recall de Positive (grieta) para alertas de seguridad.


In [ ]:
modelo_docente.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        preds = modelo_docente(images).argmax(1).cpu().numpy()
        y_pred.extend(preds.tolist())
        y_true.extend(labels.numpy().tolist())
acc_val = accuracy_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicho'); ax.set_ylabel('Real'); ax.set_title('Matriz de confusión (val)')
plt.tight_layout()
plt.show()
print(classification_report(y_true, y_pred, target_names=class_names))
print(f"Accuracy validación: {acc_val:.3f}")


In [ ]:
# --- Autoevaluación 10 ---
# Requiere (celda «Aquí coloca tu solución»): `acc_val`, `cm`
r = []
try:
    r = verificar_metricas(acc_val, cm)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('10 — Métricas', r)


## Pregunta 11 — Casos locales

### 📘 Subpreguntas
- **11.a** ¿La CNN mira la grieta o artefactos (sombra, mancha)?
- **11.b** ¿Qué harías para auditar un falso positivo?


#### ✅ Reflexión 9

Casos locales revelan si el modelo usa contexto o artefactos.


In [ ]:
N_CASOS_MOSTRADOS = 3
modelo_docente.eval()
fig, axes = plt.subplots(1, N_CASOS_MOSTRADOS, figsize=(3 * N_CASOS_MOSTRADOS, 3))
if N_CASOS_MOSTRADOS == 1:
    axes = [axes]
for i in range(N_CASOS_MOSTRADOS):
    img, label = val_ds[i]
    with torch.no_grad():
        pred = modelo(img.unsqueeze(0).to(device)).argmax(1).item()
    vis = img.numpy().transpose(1, 2, 0) * 0.5 + 0.5
    axes[i].imshow(vis.clip(0, 1))
    axes[i].set_title(f"real: {class_names[label]} | pred: {class_names[pred]}")
    axes[i].axis('off')
plt.tight_layout()
plt.show()


In [ ]:
# --- Autoevaluación 11 ---
# Requiere (celda «Aquí coloca tu solución»): `N_CASOS_MOSTRADOS`
r = []
try:
    r = verificar_casos_locales(N_CASOS_MOSTRADOS)
except NameError as err:
    print(f"❌ Ejecuta primero tu celda de solución. Falta: {err}")
    r = [False]
resumen_seccion('11 — Casos locales', r)


## Pregunta 12 — Reflexión ingeniería

### 📘 Subpreguntas
- **10.a** ¿Dónde desplegarías este modelo en un puente o edificio?
- **10.b** ¿Qué datos adicionales recolectarías en obra?
- **10.c** ¿La CNN sustituye la inspección normativa?


#### ✅ Respuesta sugerida

- **Triaje** de fotos de dron o cámaras fijas; no sustituye perito ni NDT.
- **Falsos positivos** por manchas/humedad; **falsos negativos** en grietas finas.
- Validar con inspección visual, repetir bajo distinta iluminación y documentar en informe.


## Cierre docente

EDA visual + augmentation + CNN: en obra combina triaje automático con inspección humana.
